# 01 — Embeddings: From Zero to Semantic Search
### Cortex-RAG Series | github.com/ather-ops
---
**What you will learn in this notebook:**
- What is an embedding — the math behind it
- One-hot encoding vs dense vectors
- Embedding matrix multiplication from scratch
- Cosine similarity — measure meaning not just words
- SentenceTransformer — real pretrained embeddings
- Semantic search on real data
- Save embeddings to CSV for later use in RAG

> This notebook is the **backbone** of every RAG system. Master this first.

## What is an Embedding?

An embedding converts text into a list of numbers.

These numbers **capture meaning** — not just spelling.

| Word | Normal representation | Embedding |
|------|----------------------|-----------|
| "King" | Random index | [0.2, 0.8, 0.1, ...] |
| "Queen" | Random index | [0.2, 0.7, 0.2, ...] |
| "Pizza" | Random index | [0.9, 0.1, 0.8, ...] |

King and Queen have **similar numbers** because they have similar meaning.
King and Pizza have **different numbers** because they mean different things.

**This is how machines understand language.**

## Step 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully")

## Step 2 — Level 0: One-Hot Encoding (The Old Way)

Before embeddings, computers used one-hot encoding.
Every word got a vector with all zeros except one position.

**Problem:** No meaning captured. King and Queen look completely unrelated.

In [ ]:
print("="*50)
print("LEVEL 0 — One-Hot Encoding")
print("="*50)

vocabulary = ["king", "queen", "pizza", "burger"]
word_to_index = {w:i for i,w in enumerate(vocabulary)}

def one_hot(word, vocab):
    vec = np.zeros(len(vocab))
    vec[vocab[word]] = 1
    return vec

king_oh   = one_hot("king",   word_to_index)
queen_oh  = one_hot("queen",  word_to_index)
pizza_oh  = one_hot("pizza",  word_to_index)

print(f"king  : {king_oh}")
print(f"queen : {queen_oh}")
print(f"pizza : {pizza_oh}")

# Dot product similarity
sim_king_queen = np.dot(king_oh, queen_oh)
sim_king_pizza = np.dot(king_oh, pizza_oh)

print(f"\nSimilarity king-queen : {sim_king_queen}")
print(f"Similarity king-pizza : {sim_king_pizza}")
print("Problem: Both show 0 — one-hot cannot measure meaning")

## Step 3 — Level 1: Manual Embedding Matrix

We fix the one-hot problem by multiplying with an **embedding matrix**.

The embedding matrix maps each word to a dense vector with actual values.
These values are learned during training — but here we set them manually to understand the concept.

In [ ]:
print("="*50)
print("LEVEL 1 — Manual Embedding Matrix")
print("="*50)

# 4 words, embedding dimension = 3
# Each row = embedding for one word
# Rows: king, queen, pizza, burger
E = np.array([
    [0.8, 0.9, 0.1],   # king  — high royalty, low food
    [0.8, 0.8, 0.1],   # queen — high royalty, low food
    [0.1, 0.1, 0.9],   # pizza — low royalty, high food
    [0.1, 0.2, 0.8],   # burger— low royalty, high food
])

print("Embedding matrix shape:", E.shape)
print("Rows = words, Columns = learned features")
print()

# Get embedding for 'king' using one-hot
king_emb   = np.dot(one_hot("king",   word_to_index), E)
queen_emb  = np.dot(one_hot("queen",  word_to_index), E)
pizza_emb  = np.dot(one_hot("pizza",  word_to_index), E)

print(f"king  embedding: {king_emb}")
print(f"queen embedding: {queen_emb}")
print(f"pizza embedding: {pizza_emb}")

## Step 4 — Cosine Similarity from Scratch

In [ ]:
print("="*50)
print("COSINE SIMILARITY — From Scratch")
print("="*50)

def cosine_sim(v1, v2):
    dot    = np.dot(v1, v2)
    norm1  = np.linalg.norm(v1)
    norm2  = np.linalg.norm(v2)
    return dot / (norm1 * norm2)

sim_kq = cosine_sim(king_emb, queen_emb)
sim_kp = cosine_sim(king_emb, pizza_emb)

print(f"Cosine similarity king  <-> queen : {sim_kq:.4f}  (HIGH — same meaning group)")
print(f"Cosine similarity king  <-> pizza : {sim_kp:.4f}  (LOW  — different meaning)")
print()
print("Range: -1 (opposite) to 0 (unrelated) to 1 (identical)")
print("king and queen are CLOSE because they share semantic features")

## Step 5 — Visualize Embeddings in 2D

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0a14')

words_list = ["king", "queen", "pizza", "burger"]
colors = ['#f97316', '#f97316', '#06b6d4', '#06b6d4']
embeddings_2d = E[:, :2]  # Use first 2 dims for visualization

ax = axes[0]
ax.set_facecolor('#0f172a')
for i, (word, color) in enumerate(zip(words_list, colors)):
    ax.scatter(embeddings_2d[i,0], embeddings_2d[i,1],
               color=color, s=200, zorder=5)
    ax.annotate(word, (embeddings_2d[i,0]+0.01, embeddings_2d[i,1]+0.01),
                color='white', fontsize=12, fontweight='bold')
ax.set_title('Word Embeddings in 2D Space', color='white', fontsize=13)
ax.set_xlabel('Feature 1 (Royalty)', color='#94a3b8')
ax.set_ylabel('Feature 2 (Food)', color='#94a3b8')
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values(): spine.set_color('#1e293b')
ax.annotate('', xy=embeddings_2d[1,:2], xytext=embeddings_2d[0,:2],
            arrowprops=dict(arrowstyle='->', color='#f97316', lw=2))
ax.annotate('', xy=embeddings_2d[3,:2], xytext=embeddings_2d[2,:2],
            arrowprops=dict(arrowstyle='->', color='#06b6d4', lw=2))

# Similarity heatmap
sim_matrix = np.array([[cosine_sim(E[i], E[j]) for j in range(4)] for i in range(4)])
ax2 = axes[1]
ax2.set_facecolor('#0f172a')
im = ax2.imshow(sim_matrix, cmap='Oranges', vmin=0, vmax=1)
ax2.set_xticks(range(4)); ax2.set_yticks(range(4))
ax2.set_xticklabels(words_list, color='white')
ax2.set_yticklabels(words_list, color='white')
for i in range(4):
    for j in range(4):
        ax2.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                 color='black', fontsize=11, fontweight='bold')
ax2.set_title('Cosine Similarity Heatmap', color='white', fontsize=13)
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.savefig('embedding_visualization.png', dpi=120, bbox_inches='tight', facecolor='#0a0a14')
plt.show()
print("Observation: king-queen and pizza-burger are similar pairs")

## Step 6 — SentenceTransformer: Real Pretrained Embeddings

In [ ]:
print("="*50)
print("REAL EMBEDDINGS — SentenceTransformer")
print("="*50)

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "I love machine learning",
    "Machine learning is amazing",
    "I hate vegetables",
    "Pizza tastes great",
    "Deep learning is a part of AI"
]

embeddings = model.encode(sentences)
print(f"Shape: {embeddings.shape}")
print(f"Each sentence = vector of {embeddings.shape[1]} numbers")

sim = cosine_similarity(embeddings)
print("\nSimilarity matrix:")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s[:40]}")
print()
print(sim.round(3))

## Step 7 — Semantic Search Function

In [ ]:
def semantic_search(query, corpus, model, top_k=3):
    corpus_embeddings = model.encode(corpus)
    query_embedding   = model.encode([query])
    scores = cosine_similarity(query_embedding, corpus_embeddings).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    print(f"Query: {query}")
    print("-"*50)
    for i, idx in enumerate(top_indices):
        print(f"  {i+1}. Score: {scores[idx]:.4f} | {corpus[idx]}")
    print()

corpus = [
    "Python is great for machine learning",
    "I love cooking pasta and pizza",
    "Neural networks learn from data",
    "The weather is sunny today",
    "Gradient descent optimizes ML models",
    "Football is a popular sport",
    "Deep learning uses multiple layers",
    "I enjoy eating Indian food"
]

semantic_search("What is machine learning", corpus, model)
semantic_search("Tell me about food",        corpus, model)
semantic_search("How do neural networks work", corpus, model)

## Step 8 — Save Embeddings to CSV

In [ ]:
print("="*50)
print("SAVING EMBEDDINGS TO CSV")
print("="*50)

sample_data = pd.DataFrame({
    'text': [
        "Python is great for machine learning",
        "I love cooking pasta and pizza",
        "Neural networks learn from data",
        "The weather is sunny today",
        "Gradient descent optimizes ML models",
    ]
})

embs = model.encode(sample_data['text'].tolist())

# Save embeddings as columns
emb_df = pd.DataFrame(embs, columns=[f'emb_{i}' for i in range(embs.shape[1])])
final_df = pd.concat([sample_data, emb_df], axis=1)
final_df.to_csv('embeddings_saved.csv', index=False)

print(f"Saved {len(final_df)} rows")
print(f"Shape: {final_df.shape}")
print(f"Columns: text + {embs.shape[1]} embedding dimensions")
print()

# Reload and verify
loaded = pd.read_csv('embeddings_saved.csv')
emb_cols = [c for c in loaded.columns if c.startswith('emb_')]
loaded_embs = loaded[emb_cols].values
print(f"Reloaded shape: {loaded_embs.shape}")
print("Embeddings saved and reloaded successfully")

## Summary — What You Learned

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| One-Hot Encoding | Each word = sparse vector | Cannot capture meaning |
| Embedding Matrix | Dense vector per word | Captures semantic relationships |
| Cosine Similarity | Angle between vectors | Measures meaning similarity |
| SentenceTransformer | Pretrained 384-dim embeddings | State of the art sentence meaning |
| Semantic Search | Query → find similar text | Core of every RAG system |
| Save to CSV | Persist embeddings | Reuse without recomputing |

---
**Next Notebook:** Full Netflix Pipeline with EDA + Semantic Search
**After that:** Vector Databases (Chroma) + LLM = Complete RAG System

Built by Ather Assadullah — github.com/ather-ops/Cortex-RAG